In [8]:
# Importación de bibliotecas estándar para análisis de datos
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ajuste de parámetros globales para que las gráficas sean legibles y estéticas
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [ ]:
from housing.config import SPLIT_DIR

df = pd.read_csv(SPLIT_DIR / "housing_train_raw.csv")



In [3]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,NEAR BAY
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,<1H OCEAN
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,INLAND
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,INLAND
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,NEAR OCEAN


In [4]:
df_train = df.drop("median_house_value", axis=1)
df_labels = df["median_house_value"].copy()

In [5]:
print(type(df_train))
print(type(df_labels))

<class 'pandas.DataFrame'>
<class 'pandas.Series'>


In [16]:
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, TransformerMixin


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmans_.cluster_centers_, gamma=self.gamma)
    
    def ger_feautre_name_out(self, name=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

### Creacion del pipeline

- **bedrooms / rooms_per_house / people_per_house (ratio_pipeline):** crea 3 features nuevas calculando proporciones entre columnas (ratios) para capturar relaciones más útiles que los valores crudos.

- **log (log_pipeline):** aplica transformación logarítmica a variables con “cola larga” para reducir sesgo, comprimir outliers y hacer la distribución más amigable para el modelo.

- **geo (cluster_simil):** convierte lat/long en features de similitud/distancia a clusters geográficos, para que el modelo entienda mejor “zonas” sin aprender geografía desde cero.

- **cat (cat_pipeline):** toma columnas categóricas (object), rellena faltantes y las convierte a número


In [ ]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline

# Activar el modo diagrama de pipeline
from sklearn import set_config

# Activar el modo diagrama de pipeline
set_config(display="diagram")

# Toma 2 columnas (X[:,0] y X[:,1] y devuelve el ratioo col0/col1 (una sola columna))
def column_ratio(X):
    return X[:, [0] / X[:, [1]]]

# skelearn llama esta funcion para nombrar las columnas de salida del FunctionTransformer
# (no la llamas tu). Aqui siempre devolvemos un solo nombre: "ratio"
def ratio_name(function_transformer, feature_name_in):
    return ["ratio"]

# Pipeline para crear un ratio:
# 1) Imputa NaN con la mediana
# 2) Calcula el ratio (col0/col1) -> queda 1 columna
#3) Estandariza(StandardScaler)
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="meidan"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )

# Pipeline para aplicar log a columnas (una a una) y luego escalar
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

# Transformador para feutures geograficos (lat/long) basado en clusters (definido en otro lado)
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=42)

# Pipeline para las variables categoricas
cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Pieline numerico por defecto: imputar + escalar
default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# Apilca diferentes transformaciones a diferentes columnas
preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["pupulation", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "pupulation", "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline
)

#housing_prepared = preprocessing.fit_transform(df)
#housing_prepared.shape (16512, 24)
#preprocessing.get_feature_names_out()